# StatCore meta-learning: leakage-free model combination, sklearn adapters and metrics

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-meta.ipynb)

**Paper topic:** StatCore Meta-Learning: Cross-Fitted Model Combination and NumPy Reliability Metrics — `docs/latex/statcore-meta.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-meta.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/meta_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (42, 7)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

**Cross-fitting.** $K$ stratified folds. For each base model $m$ and fold $f$, the model is fitted on the other folds and predicts fold $f$, giving out-of-fold (OOF) probabilities $P^{(m)}_i\in\Delta^{C}$ for every sample. Base models used at prediction time are **refit on all data**. Every meta-model is trained on OOF predictions only, so a base model never predicts a sample it was trained on (this limits one form of leakage; it does not remove model-selection optimism or distribution shift). Models without `predict_proba` contribute one-hot predictions. Let $\ell_m$ be the OOF log-loss of model $m$.

**`performance_aware`** — softmax weights on OOF log-loss, temperature $\tau$ (`temperature`):
$$w_m=\mathrm{softmax}_m\!\Big(-\frac{\ell_m-\min_{m'}\ell_{m'}}{\tau}\Big),\qquad P(x)=\sum_m w_m P^{(m)}(x).$$

**`metrics_based`** — features built from base-model outputs: $f1_m\cdot P^{(m)}$ (OOF macro-F1 scaled), per-model $(\max P^{(m)},\ H(P^{(m)}),\ \text{top-2 margin})$, the fraction of models agreeing with the consensus (argmax of the mean), and $H(\bar P)$; logistic-regression meta-model ($\alpha=10^{-2}$).

**`oof_stacking`** — classical stacking: logistic regression ($\alpha=10^{-2}$) on the concatenated OOF probabilities $[P^{(1)},\dots,P^{(M)}]$.

**`statistical_meta`** — features $\log\mathrm{clip}(P^{(m)},10^{-3},1)$ concatenated; the final probability is the **average of three** meta-models: shrinkage LDA, a *cluster posterior* (k-means++ with $k=\max(C,\min(2C,\lfloor n/5\rfloor))$ clusters on the meta-features, class posterior per cluster with add-one smoothing), and logistic regression.

**`calibrated_stack`** — stacking logistic regression whose logits are divided by a temperature $T=e^{\tau}$, $\tau\in[-3,3]$, learned by minimising NLL on *nested* out-of-fold meta predictions ($\min(K,3)$ inner folds): $P(x)=\mathrm{softmax}(\text{logits}(x)/T)$.

### Evaluation metrics (`ml_reliability.statcore.evaluation.metrics`)
Accuracy; macro precision/recall/F1 over the union of classes; multiclass MCC (Gorodkin): $\mathrm{MCC}=\frac{c\,n-\sum_k p_kt_k}{\sqrt{(n^2-\sum p_k^2)(n^2-\sum t_k^2)}}$; log-loss $-\frac1n\sum_i\log\max(p_{i,y_i},10^{-12})$; ECE over 10 equal-width confidence bins $\sum_b\frac{|B_b|}{n}\big|\mathrm{acc}(B_b)-\mathrm{conf}(B_b)\big|$; regression MSE/MAE/$R^2$; reconstruction MSE/MAE/relative Frobenius error; clustering inertia and number of clusters found.

### sklearn-compatible adapters
`StatCoreLogisticClassifier`, `StatCoreLDAClassifier`, `StatCoreKNNClassifier`, `StatCoreMLPClassifier`, `StatCoreMetaClassifier` (in `ml_reliability`) inherit `BaseEstimator`/`ClassifierMixin` (parameters are constructor fields → `clone`, `GridSearchCV`, pipelines work), validate input, expose `classes_`, `predict_proba`, `predict`; all numerics are done by StatCore. Only the *adapters* import scikit-learn.

## 3. Experiment (mirrors `meta_experiment.py`)

Base learners (all StatCore): logistic (`alpha=1e-2`), LDA (`shrinkage`), k-NN ($k=5$). 25 % stratified held-out split, 3-fold OOF. Reference baseline: sklearn `StackingClassifier` over sklearn logistic / LDA / k-NN with a logistic final estimator (`cv=3`).

In [ ]:
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import StackingClassifier                        # baseline only
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ml_reliability.statcore import KNNClassifier, LDA, LogisticRegression, MetaLearner
from ml_reliability.statcore.evaluation.metrics import classification_metrics
from ml_reliability.statcore.meta.meta_learning import VARIANTS

def datasets():
    for name, loader in (("wine", load_wine), ("breast_cancer", load_breast_cancer)):
        bunch = loader()
        yield name, np.asarray(bunch.data, dtype=float), np.asarray(bunch.target)

def base_models():
    return [("logistic", LogisticRegression(alpha=1e-2, random_state=0)),
            ("lda", LDA(variant="shrinkage", random_state=0)),
            ("knn", KNNClassifier(n_neighbors=5, variant="base", random_state=0))]

def sklearn_stack():
    return StackingClassifier(
        estimators=[("logistic", make_pipeline(StandardScaler(), SkLogisticRegression(max_iter=1500))),
                    ("lda", LinearDiscriminantAnalysis()), ("knn", KNeighborsClassifier(n_neighbors=5))],
        final_estimator=SkLogisticRegression(max_iter=1500), stack_method="predict_proba", cv=3, n_jobs=1)

rows = []
for dataset, X, y in datasets():
    for seed in SEEDS:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
        models = {v: MetaLearner(base_models(), meta_variant=v, n_folds=3, random_state=seed).fit(Xtr, ytr) for v in VARIANTS}
        # individual StatCore base learners, for reference
        for name, est in base_models():
            models[f"base:{name}"] = est.fit(Xtr, ytr)
        models["sklearn_stacking_baseline"] = sklearn_stack().fit(Xtr, ytr)
        for name, mdl in models.items():
            rows.append(dict(dataset=dataset, seed=seed, model=name,
                             **classification_metrics(yte, mdl.predict(Xte), mdl.predict_proba(Xte), classes=mdl.classes_)))
res = pd.DataFrame(rows)
ORDER = list(res.model.unique())
cols = ["accuracy", "f1_macro", "mcc", "log_loss", "ece"]
print(f"Held-out metrics, mean over seeds {SEEDS} (25 % stratified test split)")
mean = res.groupby(["dataset", "model"], sort=False)[cols].mean()
mean.round(4)

In [ ]:
print("sample SD over seeds"); print(res.groupby(["dataset", "model"], sort=False)[cols].std().round(4).to_string())
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, metric in zip(axes, ("log_loss", "accuracy")):
    mean[metric].unstack("dataset").loc[ORDER].plot.barh(ax=ax); ax.set_title(metric); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 4. Inside the meta-learners

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(*load_wine(return_X_y=True), test_size=0.25, random_state=SEEDS[0], stratify=load_wine().target)
pa = MetaLearner(base_models(), meta_variant="performance_aware", n_folds=3, random_state=SEEDS[0]).fit(Xtr, ytr)
print("OOF log-loss per base model:", {k: round(v, 4) for k, v in pa.oof_log_loss_.items()})
print("performance_aware weights:  ", dict(zip([n for n, _ in pa.base_estimators], np.round(pa.weights_, 4).tolist())), "| best:", pa.best_estimator_)
cs = MetaLearner(base_models(), meta_variant="calibrated_stack", n_folds=3, random_state=SEEDS[0]).fit(Xtr, ytr)
print(f"calibrated_stack learned temperature T = {cs.temperature_:.3f}")
sm = MetaLearner(base_models(), meta_variant="statistical_meta", n_folds=3, random_state=SEEDS[0]).fit(Xtr, ytr)
print("statistical_meta cluster-posterior shape (clusters x classes):", sm.meta_model_["cluster_posterior"].shape)
print("OOF accuracy per base model:", {k: round(v, 3) for k, v in pa.performance_scores_.items()})
print("evaluate() on held-out data:", {k: round(v, 4) for k, v in cs.evaluate(Xte, yte).items()})

## 5. sklearn-compatible adapters

The adapters plug StatCore models into the scikit-learn ecosystem (cross-validation, `clone`, grid search, pipelines). Here scikit-learn provides the *tooling* only; the models being evaluated are StatCore's.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score
from ml_reliability import (StatCoreKNNClassifier, StatCoreLDAClassifier, StatCoreLogisticClassifier,
                            StatCoreMetaClassifier, StatCoreMLPClassifier)
from ml_reliability.statcore_estimators import STATCORE_VARIANTS

print("own variants available to the adapters:", STATCORE_VARIANTS)
X, y = load_wine(return_X_y=True)
adapters = {
    "StatCoreLogisticClassifier": StatCoreLogisticClassifier(variant="stat_regularised"),
    "StatCoreLDAClassifier": StatCoreLDAClassifier(variant="shrinkage"),
    "StatCoreKNNClassifier": StatCoreKNNClassifier(variant="stat_k_selection"),
    "StatCoreMLPClassifier": StatCoreMLPClassifier(variant="stat_regularised", max_epochs=60),
    "StatCoreMetaClassifier": StatCoreMetaClassifier(meta_variant="calibrated_stack", n_folds=3),
}
rows = []
for name, est in adapters.items():
    scores = cross_val_score(clone(est), X, y, cv=3, scoring="accuracy")
    rows.append(dict(adapter=name, cv_accuracy=scores.mean(), sd=scores.std(ddof=1)))
pd.DataFrame(rows).round(4)

In [ ]:
est = StatCoreMetaClassifier(meta_variant="oof_stacking", n_folds=3, random_state=0).fit(Xtr, ytr)
print("classes_:", est.classes_, "| predict_proba rows sum to 1:", np.allclose(est.predict_proba(Xte).sum(1), 1.0))
print("get_params keys:", sorted(est.get_params())[:5], "...")

grid = GridSearchCV(StatCoreLogisticClassifier(), {"variant": list(STATCORE_VARIANTS["logistic"]) + ["base"], "alpha": [1e-3, 1e-1]}, cv=3)
grid.fit(Xtr, ytr)
print("GridSearchCV best:", grid.best_params_, "cv acc =", round(grid.best_score_, 4), "| held-out acc =", round(grid.score(Xte, yte), 4))

## 6. The evaluation metrics on their own

All five metric families are plain NumPy functions usable with any model.

In [ ]:
from ml_reliability.statcore import (calibration_metrics, classification_metrics, clustering_metrics,
                                      reconstruction_metrics, regression_metrics, KMeans, PCA, RidgeRegression)
proba = est.predict_proba(Xte); pred = est.classes_[proba.argmax(1)]
print("classification:", {k: round(v, 4) for k, v in classification_metrics(yte, pred, proba, classes=est.classes_).items()})
print("calibration:   ", {k: round(v, 4) for k, v in calibration_metrics(yte, proba, n_bins=10, classes=est.classes_).items()})
Xs = (X - X.mean(0)) / X.std(0)
reg = RidgeRegression().fit(Xs[:, 1:], Xs[:, 0])
print("regression:    ", {k: round(v, 4) for k, v in regression_metrics(Xs[:, 0], reg.predict(Xs[:, 1:])).items()})
pca = PCA(n_components=2).fit(Xs)
print("reconstruction:", {k: round(v, 4) for k, v in reconstruction_metrics(Xs, pca.inverse_transform(pca.transform(Xs))).items()})
km = KMeans(n_clusters=3, variant="better_init", random_state=0).fit(Xs)
print("clustering:    ", clustering_metrics(Xs, km.labels_, km.cluster_centers_))

## 7. Summary and limitations

In [ ]:
summary = mean["log_loss"].unstack("dataset").loc[ORDER].round(4)
summary["rank_wine"] = summary["wine"].rank(); summary["rank_breast_cancer"] = summary["breast_cancer"].rank()
print("Held-out log-loss (lower is better) and ranks:"); print(summary.to_string())

**What to read from the tables.** In our local 2-seed run `calibrated_stack` had the lowest held-out log-loss on both datasets (ahead of sklearn's `StackingClassifier`), `oof_stacking` was close, and `statistical_meta`/`metrics_based` were worse on wine. Note that the shrinkage-LDA base learner is poor on these *unscaled* features (OOF accuracy ≈ 0.77 on wine), which is why `performance_aware` puts almost all weight on the logistic model: stacking is only as good as its base learners.

**Honest limitations**
- Both datasets are small and nearly separable (wine 178 rows, breast cancer 569), accuracy is close to ceiling for every method, and only two seeds are used here (the paper reports a single seed 42): the ranking among meta variants is **not statistically established**.
- The sklearn baseline uses default `LinearDiscriminantAnalysis` and a different logistic regularisation, so the comparison is "natural defaults on each side", not an identical-hyper-parameter test.
- OOF stacking removes one leakage path but not model-selection optimism: the number of folds, base learners and variant were chosen without nested evaluation here. Cross-fitting costs $K+1$ fits per base model.
- `statistical_meta` and `metrics_based` have many meta-features relative to $n$ on small data; `calibrated_stack` learns one scalar from OOF logits that are themselves noisy at small $n$.
- Metrics: ECE with 10 bins on ~45–143 test rows is very noisy; MCC/F1 are reported on a single split. None of this covers distribution shift.
- The adapters need scikit-learn at import time (the numeric core does not); `MLP` adapters are comparatively slow.